# 03 — Exploration PCA, stabilité et shortlist (tâches 15–16)

Ce notebook utilise exclusivement les objets QC acceptés des batches 1–2 et les matrices/prétraitements techniquement acceptés par le notebook 02. Les batches 3–4 ne sont ni ajustés, ni projetés, ni évalués ici.

La sélection porte uniquement sur les prétraitements, séparément pour `object_matrix` et `pixel_matrix`. Après la revue humaine, chaque prétraitement doit rester admissible sur toutes les variantes de matrice attendues ; ses métriques sont agrégées en pire cas, puis tous les prétraitements du front de Pareto sont conservés, sans score pondéré, filtre de diversité ni plafond actif.


## 1. Imports et contrats 

In [1]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = CURRENT_DIR if (CURRENT_DIR / "src").is_dir() else CURRENT_DIR.parent
if not (PROJECT_ROOT / "src").is_dir():
    raise RuntimeError("Launch the notebook from the project root or notebooks/.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src import experiment_config as expcfg
from src.io.database_h5 import load_nir_uco_h5
from src.protocol_governance import sha256_file, sha256_payload, verify_frozen_protocol
from src.spectra.band_selection import select_wavelength_range_from_database
from src.utils import save_parquet, load_parquet
from src.workflows.matrix_preprocessing import wavelength_axis_id
from src.workflows.pca import (
    build_pca_candidate_plan,
    evaluate_pca_stability,
    fit_pca_candidate,
    subset_object_db_for_pca,
    summarize_pca_stability,
)
from src.workflows.pca_selection import (
    apply_pca_artifact_review_decisions,
    build_pca_artifact_review_table,
    build_pca_run_fingerprint,
    build_pca_scoring_diagnostics,
    build_pca_selection_audit,
    build_pca_selection_diagnostics,
    freeze_pca_shortlist,
    hash_pca_input_artifacts,
    hash_pca_review_table,
    make_pca_selection_config,
    pca_input_artifact_paths,
    pca_input_fingerprint,
    select_pca_preprocessing_shortlist,
    validate_pca_artifact_review,
    validate_pca_preprocessing_shortlist,
)
from src.workflows.protocol_audit import assert_no_forbidden_score_columns, assert_pca_selection_audit_consistency
from src.workflows.protocol_split import build_grouped_folds, eligible_object_ids
from src.visualization.plot_pca import build_pca_visual_review_pdf

pd.set_option("display.max_columns", 60)
pd.set_option("display.max_rows", expcfg.PCA_MAX_ROWS_TO_DISPLAY)
if "shortlist_id" in expcfg.PCA_SELECTED_PREPROCESSING_COLUMNS:
    raise RuntimeError(
        "Legacy 'shortlist_id' is still present in "
        "PCA_SELECTED_PREPROCESSING_COLUMNS. "
        "Remove it from experiment_config.py: selection_set_id is now "
        "the unique frozen-set identifier."
    )

## 2. Protocole gelé et artefacts d’entrée

In [2]:
RESULTS_TAG = (
    f"{int(expcfg.WAVELENGTH_WINDOW_MIN_NM)}_{int(expcfg.WAVELENGTH_WINDOW_MAX_NM)}"
    if expcfg.USE_WAVELENGTH_WINDOW
    else expcfg.DEFAULT_RESULTS_TAG
)
PROTOCOL_DIR = PROJECT_ROOT.joinpath(*expcfg.PROTOCOL_ARTIFACT_RELATIVE_DIR)
protocol_checks_df = verify_frozen_protocol(PROTOCOL_DIR, strict=True)
protocol_lock = json.loads(
    (PROTOCOL_DIR / expcfg.PROTOCOL_OUTPUT_FILENAMES["lock"]).read_text(encoding="utf-8")
)
PROTOCOL_HASH = str(protocol_lock["lock_sha256"])

input_paths = pca_input_artifact_paths(PROJECT_ROOT, results_tag=RESULTS_TAG)
input_hashes = hash_pca_input_artifacts(PROJECT_ROOT, results_tag=RESULTS_TAG)
INPUT_FINGERPRINT = pca_input_fingerprint(input_hashes)
matrix_summary_df = load_parquet(input_paths["matrix_summary"])
m_feasibility_df = load_parquet(input_paths["m_feasibility"])
preprocessing_validation_df = load_parquet(input_paths["preprocessing_validation"])
wavelength_config_df = load_parquet(input_paths["wavelength_config"])
split_manifest_df = load_parquet(input_paths["protocol_split_manifest"])

if not preprocessing_validation_df["fit_role"].eq("calibration").all():
    raise RuntimeError("Notebook-03 input contains non-calibration preprocessing fits")

if not preprocessing_validation_df["eval_role"].eq("calibration").all():
    raise RuntimeError(
            "preprocessing_validation.parquet is not calibration-only. "
            "Rerun the updated notebook 02 before notebook 03."
        )

RESULTS_DIR = PROJECT_ROOT / "results" / f"{expcfg.PCA_RESULTS_DIR_PREFIX}_{RESULTS_TAG}"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_PATHS = {
    name: RESULTS_DIR / filename
    for name, filename in expcfg.PCA_OUTPUT_FILENAMES.items()
}
display(protocol_checks_df)
display(pd.DataFrame({"artifact": input_hashes.keys(), "sha256": input_hashes.values()}))
print("PROTOCOL_HASH:", PROTOCOL_HASH)
print("INPUT_FINGERPRINT:", INPUT_FINGERPRINT)

,check,passed,detail
0,all_frozen_artifacts_exist,True,missing=[]
1,configuration_sha256_matches_current_protocol,True,expected=9e87eac5b065a9fae9ef1ff543981234bfbda...
2,inference_plan_sha256_matches_current_protocol,True,expected=10624351f77a2b18a37b8a51b766be759e4cc...
3,planned_contrasts_sha256_matches_current_protocol,True,expected=5a557c1c13441366f7795cbed2504f70516af...
4,checks_file_checksum_matches_lock,True,expected=9e972be2cfa8cfe634a304b706c12b9cab103...
5,inference_plan_file_checksum_matches_lock,True,expected=3b34fa6331ce14de3bb2b63202cbb821333cc...
6,manifest_file_checksum_matches_lock,True,expected=80fd7411e9a0493d37f60ec59c229702ee3b4...
7,planned_contrasts_file_checksum_matches_lock,True,expected=14abc9529b2fa764bc9412c0122e5c981c55a...
8,lock_checksum_is_valid,True,expected=5d66e659d7da4e69fa647123058bcea08d33f...


,artifact,sha256
0,database_manifest,e83d09cbdf68206c878742396864941a65247c36e7dbd4...
1,protocol_split_manifest,57188a7cadc29bbfad767bcadf558bd5917754a77b3edf...
2,wavelength_config,d7f1f4d4abb9a3b399fe9fd27027e8ffa71a0abe107e2e...
3,matrix_summary,ab6bf26d1808438c79a7c2fa30e0b2a27ad68433b437ae...
4,m_feasibility,9d3c19e9b6d2cc0fd0c80021672483b5abec7af74cb997...
5,preprocessing_validation,9311cffc14706b43c716698e464d70039dfb2a595dacee...


PROTOCOL_HASH: 5d66e659d7da4e69fa647123058bcea08d33fe0155b6c59d71001820dbc78f9e
INPUT_FINGERPRINT: b8f999522dcbf75126a023c6516e6f78a70c99882c20d186575e62e69d3cb297


## 3. Axe spectral, univers de candidats et folds communs

In [3]:
object_db, image_db = load_nir_uco_h5(
    PROJECT_ROOT.joinpath(*expcfg.DATABASE_H5_RELATIVE_PATH),
    reconstruct_heavy_object_arrays=False,
    batches=expcfg.PCA_CALIBRATION_BATCHES,
)
# PCA only consumes object spectra and wavelength axes. Releasing image
# arrays here avoids retaining or copying multi-gigabyte cubes.
for image in image_db.values():
    for field in ("cube", "image_ref", "mask", "labels"):
        image.pop(field, None)
if expcfg.USE_WAVELENGTH_WINDOW:
    object_db, image_db, wavelengths, _ = select_wavelength_range_from_database(
        object_db=object_db,
        image_db=image_db,
        min_nm=expcfg.WAVELENGTH_WINDOW_MIN_NM,
        max_nm=expcfg.WAVELENGTH_WINDOW_MAX_NM,
    )
else:
    wavelengths = np.asarray(next(iter(object_db.values()))["wavelengths"], dtype=float)
if len(wavelength_config_df) != 1:
    raise RuntimeError(
        f"Notebook 02 wavelength configuration has {len(wavelength_config_df)} rows, expected 1."
    )

locked_axis_id = str(wavelength_config_df.iloc[0]["wavelength_axis_id"])
if not bool(wavelength_config_df.iloc[0]["locked"]):
    raise RuntimeError("Notebook 02 wavelength configuration is not locked.")
if wavelength_axis_id(wavelengths) != locked_axis_id:
    raise RuntimeError("Database wavelength axis differs from the notebook-02 lock.")

candidate_plan_df = build_pca_candidate_plan(
    matrix_summary_df,
    m_feasibility_df,
    preprocessing_validation_df,
    allowed_m=expcfg.PCA_BALANCED_M_VALUES,
    sg_window_length=expcfg.PCA_SG_WINDOW_LENGTH,
    matrix_methods=expcfg.PCA_MATRIX_METHODS,
    balanced_strategies=expcfg.PCA_BALANCED_STRATEGIES,
)

pca_candidate_registry_df = candidate_plan_df.loc[:,list(expcfg.PCA_CANDIDATE_REGISTRY_COLUMNS)].copy()
if pca_candidate_registry_df["candidate_id"].duplicated().any():
    raise RuntimeError("Duplicate PCA candidate_id values.")
if pca_candidate_registry_df["selection_unit_id"].isna().any():
    raise RuntimeError("At least one PCA candidate has no selection_unit_id.")
save_parquet(pca_candidate_registry_df, OUTPUT_PATHS["candidate_registry"])

RUN_FINGERPRINT = build_pca_run_fingerprint(
    candidate_plan_df,
    protocol_hash=PROTOCOL_HASH,
    input_hashes=input_hashes,
)
PCA_COMPUTE_FINGERPRINT = sha256_payload({
    "run_fingerprint": RUN_FINGERPRINT,
    "source_hashes": {
        path: sha256_file(PROJECT_ROOT / path)
        for path in (
            "src/workflows/pca.py",
            "src/workflows/pca_selection.py",
            "src/matrices/matrix_registry.py",
            "src/matrices/redim_matrix.py",
            "src/spectra/preprocessing.py",
        )
    },
})
CHECKPOINT_PATHS = {
    "diagnostics": RESULTS_DIR / ".pca_candidate_diagnostics.parquet",
    "components": RESULTS_DIR / ".pca_components.parquet",
}

calibration_object_ids = eligible_object_ids(split_manifest_df, "calibration")
object_db_calibration = subset_object_db_for_pca(
    object_db,
    sample_kind=expcfg.PCA_SAMPLE_KIND,
    reference_classes=expcfg.REFERENCE_CLASSES,
    allowed_batches=expcfg.PCA_CALIBRATION_BATCHES,
    forbidden_batches=expcfg.PCA_FORBIDDEN_BATCHES,
    allowed_object_ids=calibration_object_ids,
)

observed_batches = {int(obj["batch"]) for obj in object_db_calibration.values()}
if observed_batches != set(expcfg.PCA_CALIBRATION_BATCHES):
    raise RuntimeError(f"Unexpected PCA batches: {sorted(observed_batches)}")

calibration_manifest_df = split_manifest_df.loc[
    split_manifest_df["protocol_role"].eq("calibration")
    & split_manifest_df["qc_eligibility"].eq("accepted")
].copy()
common_folds_df, common_fold_diagnostics_df = build_grouped_folds(
    calibration_manifest_df,
    group_col=expcfg.PCA_STABILITY_GROUP_COL,
    label_col="label",
    batch_col="batch",
    n_splits=expcfg.PCA_STABILITY_N_SPLITS,
    random_state=expcfg.PCA_STABILITY_REFERENCE_SEED,
    require_complete_coverage=True,
)

candidate_counts_df = pca_candidate_registry_df.groupby(
    ["matrix_family","matrix_variant"], as_index=False
    ).agg(
        n_candidates=("candidate_id", "nunique"),
        n_selection_units=("selection_unit_id", "nunique")
    )

display(candidate_counts_df)
display(common_fold_diagnostics_df)
print("RUN_FINGERPRINT:", RUN_FINGERPRINT)

,matrix_family,matrix_variant,n_candidates,n_selection_units
0,object_matrix,object_mean,19,19
1,object_matrix,object_median,19,19
2,pixel_matrix,all_pixels,19,19
3,pixel_matrix,balanced_pixels_center_m10,19,19
4,pixel_matrix,balanced_pixels_center_m20,19,19
5,pixel_matrix,balanced_pixels_random_m10,19,19
6,pixel_matrix,balanced_pixels_random_m20,19,19


,fold_id,n_validation_groups,n_training_groups,n_validation_objects,median_validation_object_size,validation_has_all_classes,validation_has_all_batches,training_has_all_classes,training_has_all_batches,no_shared_groups,coverage_complete
0,0,2,2,104,0.0,True,True,True,True,True,True
1,1,2,2,105,0.0,True,True,True,True,True,True


RUN_FINGERPRINT: ef11b61ee355079c1fd13e01f780f601e48d56eb9be1e1a3c1f1e334ce5674b6


## 4. PCA et stabilité par candidat

In [4]:
candidate_results = {}
expected_candidate_ids = set(candidate_plan_df["candidate_id"].astype(str))
checkpoint_valid = all(path.exists() for path in CHECKPOINT_PATHS.values())
review_cache_valid = (
    OUTPUT_PATHS["artifact_review"].exists()
    and OUTPUT_PATHS["visual_review"].exists()
)
if review_cache_valid:
    cached_review = load_parquet(OUTPUT_PATHS["artifact_review"])
    cached_hashes = cached_review["review_evidence"].astype(str).str.extract(
        r";sha256=([0-9a-f]{64})$", expand=False
    )
    review_cache_valid = (
        set(cached_review["candidate_id"].astype(str)) == expected_candidate_ids
        and cached_review["run_fingerprint"].astype(str).eq(RUN_FINGERPRINT).all()
        and cached_hashes.notna().all()
        and cached_hashes.eq(sha256_file(OUTPUT_PATHS["visual_review"])).all()
    )
if checkpoint_valid:
    cached_diagnostics = load_parquet(CHECKPOINT_PATHS["diagnostics"])
    cached_components = load_parquet(CHECKPOINT_PATHS["components"])
    checkpoint_valid = all(
        frame["_compute_fingerprint"].astype(str)
        .eq(PCA_COMPUTE_FINGERPRINT).all()
        for frame in (cached_diagnostics, cached_components)
    )
use_checkpoint = checkpoint_valid and review_cache_valid
if use_checkpoint:
    print("Reusing fingerprinted PCA computation checkpoint.", flush=True)
    diagnostic_rows = cached_diagnostics.drop(
        columns="_compute_fingerprint"
    ).to_dict("records")
    pca_components_df = cached_components.drop(
        columns="_compute_fingerprint"
    )
    component_parts = [pca_components_df]
    candidate_records = []
else:
    diagnostic_rows = []
    component_parts = []
    candidate_records = candidate_plan_df.to_dict("records")
for candidate_index, candidate in enumerate(candidate_records, start=1):
    if (candidate_index == 1 or candidate_index % 10 == 0
            or candidate_index == len(candidate_records)):
        print(
            f"PCA candidate {candidate_index}/{len(candidate_records)}: "
            f"{candidate['candidate_id']}",
            flush=True,
        )
    diagnostic, result, components = fit_pca_candidate(
        object_db_calibration,
        candidate,
        n_components=expcfg.PCA_N_COMPONENTS,
        wavelengths=wavelengths,
        random_state=expcfg.PCA_STABILITY_REFERENCE_SEED,
        under_m_policy=expcfg.PCA_BALANCED_UNDER_M_POLICY,
        sg_window_length=expcfg.PCA_SG_WINDOW_LENGTH,
        sg_polyorder=expcfg.PCA_SG_POLYORDER,
        max_zero_variance_band_rate=expcfg.PREPROCESSING_MAX_ZERO_VARIANCE_BAND_RATE,
        zero_variance_epsilon=expcfg.PREPROCESSING_ZERO_VARIANCE_EPSILON,
    )
    diagnostic = dict(diagnostic)
    diagnostic["stability_valid"] = False
    if result is not None:
        candidate_results[candidate["candidate_id"]] = result
        component_parts.append(components)
        try:
            metric_stability, loading_stability = evaluate_pca_stability(
                object_db_calibration,
                candidate=candidate,
                fold_assignments=common_folds_df,
                group_col=expcfg.PCA_STABILITY_GROUP_COL,
                n_components=expcfg.PCA_STABILITY_N_COMPONENTS,
                seeds=expcfg.PCA_STABILITY_SEEDS,
                reference_seed=expcfg.PCA_STABILITY_REFERENCE_SEED,
                n_splits=expcfg.PCA_STABILITY_N_SPLITS,
                n_bootstrap=expcfg.PCA_STABILITY_N_BOOTSTRAP,
                bootstrap_group_col=expcfg.PCA_STABILITY_BOOTSTRAP_GROUP_COL,
                sg_window_length=expcfg.PCA_SG_WINDOW_LENGTH,
                sg_polyorder=expcfg.PCA_SG_POLYORDER,
                wavelengths=wavelengths,
                under_m_policy=expcfg.PCA_BALANCED_UNDER_M_POLICY,
            )
            diagnostic.update(summarize_pca_stability(metric_stability, loading_stability))
        except Exception as error:
            diagnostic["stability_valid"] = False
            diagnostic["technical_error"] = (
                str(diagnostic.get("technical_error", ""))
                + f"; stability: {type(error).__name__}: {error}"
            ).strip("; ")
    else:
        diagnostic["stability_valid"] = False
    diagnostic_rows.append(dict(diagnostic))

pca_candidate_diagnostics_df = pd.DataFrame(diagnostic_rows)
if not use_checkpoint:
    if not component_parts:
        raise RuntimeError("No PCA components were computed.")
    pca_components_df = pd.concat(component_parts, ignore_index=True)
    save_parquet(
        pca_candidate_diagnostics_df.assign(
            _compute_fingerprint=PCA_COMPUTE_FINGERPRINT
        ),
        CHECKPOINT_PATHS["diagnostics"],
    )
    save_parquet(
        pca_components_df.assign(
            _compute_fingerprint=PCA_COMPUTE_FINGERPRINT
        ),
        CHECKPOINT_PATHS["components"],
    )

diagnostic_candidate_ids = set(pca_candidate_diagnostics_df["candidate_id"].astype(str))
registry_candidate_ids = set(pca_candidate_registry_df["candidate_id"].astype(str))
if diagnostic_candidate_ids != registry_candidate_ids:
    raise RuntimeError("PCA diagnostic universe differs from the candidate registry.")

technical_columns = list(expcfg.PCA_TECHNICAL_FLAG_COLUMNS)
missing_technical_columns = [column for column in technical_columns
    if column not in pca_candidate_diagnostics_df
]
if missing_technical_columns:
    raise RuntimeError("Missing PCA technical flags: ", missing_technical_columns)

technically_valid_mask = pca_candidate_diagnostics_df[technical_columns].fillna(False).astype(bool).all(axis=1)
pca_candidate_diagnostics_df["technical_pre_review_valid"] = technically_valid_mask
review_candidates_df = pca_candidate_diagnostics_df.loc[technically_valid_mask].copy()
if review_candidates_df.empty:
    raise RuntimeError("No technically valid PCA candidate remains.")

review_candidate_ids = set(review_candidates_df["candidate_id"].astype(str))
review_plan_df = (pca_candidate_registry_df.loc[
        pca_candidate_registry_df["candidate_id"]
        .astype(str)
        .isin(review_candidate_ids)
    ]
    .copy()
)

technical_summary_df = (pca_candidate_diagnostics_df.groupby("matrix_family",as_index=False)
    .agg(
        n_candidates=("candidate_id", "size"),
        n_technical_valid=(
            "technical_pre_review_valid",
            "sum",
        ),
    )
)

technical_summary_df

PCA candidate 1/133: pca_candidate_d16d8bc240255cb1a499


PCA candidate 10/133: pca_candidate_4ee84d5c6da32b1beece


PCA candidate 20/133: pca_candidate_3467a2fb8fcacdc1bf3d


PCA candidate 30/133: pca_candidate_7f47cdb931264fbdafc9


PCA candidate 40/133: pca_candidate_a06f5bd7028c32d7f767


...

PCA candidate 110/133: pca_candidate_91c752a84cd4d7963008


PCA candidate 120/133: pca_candidate_d741a36ea86828f5bd1d


PCA candidate 130/133: pca_candidate_24c2df3c69ff2d5df547


PCA candidate 133/133: pca_candidate_04f5f605fda8cb0a6ba4


,matrix_family,n_candidates,n_technical_valid
0,object_matrix,38,38
1,pixel_matrix,95,95


## 5. Dossier visuel exhaustif et revue humaine bloquante

Le PDF est généré avant la sélection, avec une page pour chaque candidat techniquement valide. Le tableau de revue est d’abord créé ou actualisé, puis la cellule suivante applique et valide les décisions documentées. La validation bloquante intervient seulement après cette saisie.


In [5]:
existing_review_df = (
    pd.read_parquet(OUTPUT_PATHS["artifact_review"])
    if OUTPUT_PATHS["artifact_review"].exists()
    else None
)
if candidate_results:
    page_by_candidate = build_pca_visual_review_pdf(
        candidate_results,
        review_plan_df,
        OUTPUT_PATHS["visual_review"],
        wavelengths=wavelengths,
    )
    review_pdf_sha256 = sha256_file(OUTPUT_PATHS["visual_review"])
else:
    if existing_review_df is None or not OUTPUT_PATHS["visual_review"].exists():
        raise RuntimeError("PCA checkpoint has no matching visual review.")
    page_by_candidate = (
        existing_review_df.set_index("candidate_id")["review_evidence"]
        .astype(str).str.extract(r"#page=(\d+)", expand=False)
        .astype(int).to_dict()
    )
    review_pdf_sha256 = sha256_file(OUTPUT_PATHS["visual_review"])
pca_artifact_review_df = build_pca_artifact_review_table(
    review_plan_df,
    run_fingerprint=RUN_FINGERPRINT,
    review_pdf_path=str(OUTPUT_PATHS["visual_review"].resolve()),
    review_pdf_sha256=review_pdf_sha256,
    page_by_candidate=page_by_candidate,
    existing_review=existing_review_df,
).loc[:, list(expcfg.PCA_ARTIFACT_REVIEW_COLUMNS)]
save_parquet(pca_artifact_review_df, OUTPUT_PATHS["artifact_review"])

review_lookup_df = (review_plan_df[
        [
            "candidate_id",
            "selection_unit_id",
            "matrix_family",
            "matrix_variant",
            "matrix_method",
            "m",
            "balanced_pixel_strategy",
            "preprocessing",
            "preprocessing_steps",
            "sg_window_length",
            "sg_polyorder",
        ]
    ].copy()
)
review_lookup_df["pdf_page"] = review_lookup_df["candidate_id"].astype(str).map(page_by_candidate)
review_lookup_df = review_lookup_df.sort_values("pdf_page", kind="mergesort")


display(review_lookup_df.head(expcfg.PCA_MAX_ROWS_TO_DISPLAY))
print("Current visual-review PDF SHA-256:", review_pdf_sha256)
print("Review table:",OUTPUT_PATHS["artifact_review"].resolve())


,candidate_id,selection_unit_id,matrix_family,matrix_variant,matrix_method,m,balanced_pixel_strategy,preprocessing,preprocessing_steps,sg_window_length,sg_polyorder,pdf_page
0,pca_candidate_d16d8bc240255cb1a499,pca_preproc_4b685ef3705c50f960d3,object_matrix,object_mean,object_mean,NaN,not_applicable,absorbance,absorbance,NaN,NaN,1
1,pca_candidate_9caf950d96e72394836b,pca_preproc_caf14bb937ad8f269a8d,object_matrix,object_mean,object_mean,NaN,not_applicable,absorbance_msc,absorbance+msc,NaN,NaN,2
2,pca_candidate_0a0eec0228d7064bac01,pca_preproc_e5b6d626093858ae3dde,object_matrix,object_mean,object_mean,NaN,not_applicable,absorbance_sg_d1,absorbance+sg_d1,11.0,2.0,3
3,pca_candidate_81328dadd7271a8d3e29,pca_preproc_f474f06165792caceb9f,object_matrix,object_mean,object_mean,NaN,not_applicable,absorbance_sg_d2,absorbance+sg_d2,11.0,2.0,4
4,pca_candidate_5c318acb94a8a9018852,pca_preproc_3ff246c42e12fcec72dc,object_matrix,object_mean,object_mean,NaN,not_applicable,absorbance_sg_smooth,absorbance+sg_smooth,11.0,2.0,5
5,pca_candidate_f7b527837f5369fe1954,pca_preproc_c4cc4ec13b3b3c7e4b7a,object_matrix,object_mean,object_mean,NaN,not_applicable,absorbance_snv,absorbance+snv,NaN,NaN,6
6,pca_candidate_21bee3160959a0419c55,pca_preproc_a14496171a4726ed7138,object_matrix,object_mean,object_mean,NaN,not_applicable,absorbance_snv_sg_d1,absorbance+snv+sg_d1,11.0,2.0,7
7,pca_candidate_c2c0660c8aa8c08eade1,pca_preproc_7b7c6743d820a7f1a8e5,object_matrix,object_mean,object_mean,NaN,not_applicable,absorbance_snv_sg_d2,absorbance+snv+sg_d2,11.0,2.0,8
8,pca_candidate_5e564f9b9c18809dc7f6,pca_preproc_53a12e7fc341de8a6047,object_matrix,object_mean,object_mean,NaN,not_applicable,absorbance_snv_sg_smooth,absorbance+snv+sg_smooth,11.0,2.0,9
9,pca_candidate_4ee84d5c6da32b1beece,pca_preproc_85cb32cfb0ad35b78785,object_matrix,object_mean,object_mean,NaN,not_applicable,msc,msc,NaN,NaN,10


Current visual-review PDF SHA-256: 8d504d1ffeb3fa3b18fec1e960174803dbc6b1d7e3caf512c3240730fb42f9dd
Review table: C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\03_pca_8tracks_v5_px_qc_v1\pca_artifact_review.parquet


### Révision documentée du PDF courant

La cellule suivante rattache les décisions humaines au `run_fingerprint` du run
courant. Le seul verrou recopié après lecture humaine est
`REVIEWED_PDF_SHA256` : il doit être l'empreinte affichée pour le PDF relu.

Un changement du verrou de protocole n'impose donc pas une seconde lecture si
le PDF régénéré est strictement identique octet par octet. En revanche, toute
modification du PDF, de l'ordre des pages ou de l'univers des candidats bloque
la cellule et impose une nouvelle revue. Les candidats non cités dans les
groupes documentés sont explicitement classés `accept`.


In [6]:
REVIEWED_PDF_SHA256 = (
    "8d504d1ffeb3fa3b18fec1e960174803dbc6b1d7e3caf512c3240730fb42f9dd"
)
reviewer_name = "AG"
review_date = "2026-08-11"

reject_msc_outlier_dominated = {
    "pca_candidate_a06f5bd7028c32d7f767",  # p.40  all_pixels | absorbance_msc
    "pca_candidate_1271e63a08255d8940c3",  # p.48  all_pixels | msc
    "pca_candidate_6806545c4cc2f4c3e112",  # p.97  random_m10 | absorbance_msc
    "pca_candidate_1e15b6f9199457673859",  # p.105 random_m10 | msc
    "pca_candidate_535596f1d152aaf4df06",  # p.116 random_m20 | absorbance_msc
    "pca_candidate_80e16325f78147d8cec6",  # p.124 random_m20 | msc
}

warning_msc_isolated_outliers = {
    "pca_candidate_3d487c7719f88e0dfb66",  # p.67
    "pca_candidate_b1f0fa2f9d877c274255",  # p.78
    "pca_candidate_1843e5ba4ee849b63a43",  # p.86
}

warning_snv_outlier_amplification = {
    "pca_candidate_143c26de57d02210a307",  # p.44
    "pca_candidate_5e498192499bbdde3aee",  # p.53
    "pca_candidate_64c0a8d3e393595f0d54",  # p.72
    "pca_candidate_1bde718d18b47889dda1",  # p.82
    "pca_candidate_183d39e168deb8fe31d5",  # p.91
    "pca_candidate_0e37f9656507a9ca2ae5",  # p.101
    "pca_candidate_91c752a84cd4d7963008",  # p.110
    "pca_candidate_d741a36ea86828f5bd1d",  # p.120
    "pca_candidate_c84390361af443c2e675",  # p.129
}

decision_groups = (
    {
        "candidate_ids": reject_msc_outlier_dominated,
        "review_decision": "reject",
        "artifact_codes": (
            "msc_pixel_instability;"
            "score_cloud_outlier_dominated;"
            "extreme_qt2_leverage"
        ),
        "critical_artifact": True,
        "review_comment": (
            "A small number of extreme pixel observations dominates the PCA "
            "score scale and Hotelling T2 distribution, compressing the main "
            "score cloud. The latent representation is therefore visually "
            "dominated by isolated observations after MSC."
        ),
    },
    {
        "candidate_ids": warning_msc_isolated_outliers,
        "review_decision": "warning",
        "artifact_codes": (
            "msc_pixel_instability;"
            "isolated_score_outlier;"
            "moderate_qt2_outliers"
        ),
        "critical_artifact": False,
        "review_comment": (
            "One or a few isolated high-leverage observations are visible "
            "after MSC. The main score cloud nevertheless remains interpretable "
            "and is not globally dominated by these observations."
        ),
    },
    {
        "candidate_ids": warning_snv_outlier_amplification,
        "review_decision": "warning",
        "artifact_codes": (
            "snv_outlier_amplification;"
            "score_tail;"
            "moderate_qt2_outliers"
        ),
        "critical_artifact": False,
        "review_comment": (
            "SNV is associated with a visible tail of atypical observations "
            "and elevated Q/T2 values. The central score structure remains "
            "interpretable, so the candidate is retained with a warning."
        ),
    },
)

review = load_parquet(OUTPUT_PATHS["artifact_review"]).copy()
expected_candidate_ids = set(review_plan_df["candidate_id"].astype(str))
observed_candidate_ids = set(review["candidate_id"].astype(str))
if observed_candidate_ids != expected_candidate_ids:
    raise RuntimeError(
        "The review table does not match the current run: "
        f"missing="
        f"{sorted(expected_candidate_ids-observed_candidate_ids)}, "
        f"extra="
        f"{sorted(observed_candidate_ids-expected_candidate_ids)}"
    )
if not review["run_fingerprint"].astype(str).eq(RUN_FINGERPRINT).all():
    raise RuntimeError(
        "The review table is not bound to the current run_fingerprint."
    )

pca_artifact_review_df = apply_pca_artifact_review_decisions(
    review,
    decision_groups=decision_groups,
    reviewed_pdf_sha256=REVIEWED_PDF_SHA256,
    reviewer=reviewer_name,
    review_date=review_date,
    default_review_comment=(
        "Spectra and loadings are coherent; the score cloud is "
        "interpretable and not dominated by isolated observations. "
        "No critical visual artifact was identified."
    ),
).loc[:, list(expcfg.PCA_ARTIFACT_REVIEW_COLUMNS)]

validate_pca_artifact_review(
    pca_artifact_review_df,
    expected_candidate_ids=review_plan_df["candidate_id"],
    expected_run_fingerprint=RUN_FINGERPRINT,
)
save_parquet(pca_artifact_review_df, OUTPUT_PATHS["artifact_review"])

display(pca_artifact_review_df[["review_decision", "critical_artifact"]].value_counts(dropna=False))
print("Current run_fingerprint :", RUN_FINGERPRINT)
print("Reviewed PDF SHA-256 :", REVIEWED_PDF_SHA256)


review_decision  critical_artifact
accept           False                115
warning          False                 12
reject           True                   6
Name: count, dtype: int64

Current run_fingerprint : ef11b61ee355079c1fd13e01f780f601e48d56eb9be1e1a3c1f1e334ce5674b6
Reviewed PDF SHA-256 : 8d504d1ffeb3fa3b18fec1e960174803dbc6b1d7e3caf512c3240730fb42f9dd


## 7. Couverture stricte, agrégation robuste et Pareto par famille

In [7]:
selection_config = make_pca_selection_config()
pca_candidate_selection_diagnostics_df = build_pca_selection_diagnostics(
    pca_candidate_diagnostics_df,
    artifact_review_df=pca_artifact_review_df,
    config=selection_config,
)
(
    pca_shortlist_full_df,
    pca_preprocessing_summary_df,
    pca_selection_stage_summary_df,
) = select_pca_preprocessing_shortlist(
    pca_candidate_selection_diagnostics_df,
    config=selection_config,
)

review_hash = hash_pca_review_table(pca_artifact_review_df)
pca_shortlist_frozen_df = freeze_pca_shortlist(
    pca_shortlist_full_df,
    protocol_hash=PROTOCOL_HASH,
    review_hash=review_hash,
    input_hashes=input_hashes,
)

missing_selected_columns = [col for col in expcfg.PCA_SELECTED_PREPROCESSING_COLUMNS if col not in pca_shortlist_frozen_df.columns]
if missing_selected_columns:
    raise RuntimeError(
        f"Missing PCA_SELECTED_PREPROCESSING_COLUMNS in frozen shortlist: {missing_selected_columns}"
    )

pca_selected_preprocessings_df = pca_shortlist_frozen_df.loc[
    :, list(expcfg.PCA_SELECTED_PREPROCESSING_COLUMNS)
].copy()
validate_pca_preprocessing_shortlist(
    pca_selected_preprocessings_df,
    max_per_family=expcfg.MAX_PCA_PREPROCESSINGS_PER_FAMILY,
    expected_families=expcfg.PCA_SELECTION_EXPECTED_FAMILIES,
    expected_protocol_hash=PROTOCOL_HASH,
    expected_input_fingerprint=INPUT_FINGERPRINT,
    expected_review_hash=review_hash,
)

pca_scoring_diagnostics_df = build_pca_scoring_diagnostics(
    pca_candidate_selection_diagnostics_df,
    config=selection_config,
    preprocessing_summary_df=pca_preprocessing_summary_df,
).loc[:, list(expcfg.PCA_SCORING_DIAGNOSTIC_COLUMNS)]

pca_selection_audit_df = build_pca_selection_audit(
    pca_candidate_registry_df,
    pca_candidate_selection_diagnostics_df,
    pca_preprocessing_summary_df,
    config=selection_config,
).loc[:, list(expcfg.SELECTION_AUDIT_COLUMNS)]

pca_selection_audit_checks_df = assert_pca_selection_audit_consistency(
    pca_selection_audit_df,
    candidate_registry=pca_candidate_registry_df,
    preprocessing_summary=pca_preprocessing_summary_df,
    selected_preprocessings=pca_selected_preprocessings_df,
    strict=True,
)

if not component_parts:
    raise RuntimeError("No PCA components were computed.")

pca_components_df = pd.concat(component_parts, ignore_index=True)
pca_summary_df = pca_components_df.loc[:,list(expcfg.PCA_SUMMARY_COLUMNS),].copy()

audit_funnel_df = (
    pca_selection_audit_df.groupby(
        ["entity_type", "substage", "decision"],
        dropna=False,
    )
    .size()
    .reset_index(name="n_events")
    .sort_values(["entity_type", "substage", "decision"], kind="mergesort")
)

candidate_elimination_events_df = pca_selection_audit_df.loc[
        pca_selection_audit_df["entity_type"].eq("pca_candidate")
        & pca_selection_audit_df["decision"].eq("eliminated")
    ].merge(
        pca_candidate_registry_df,
        left_on="entity_id",
        right_on="candidate_id",
        how="left",
        validate="many_to_one",
    )
candidate_elimination_view = candidate_elimination_events_df[
        [
            "candidate_id",
            "selection_unit_id",
            "matrix_family",
            "matrix_variant",
            "matrix_method",
            "m",
            "balanced_pixel_strategy",
            "preprocessing",
            "sg_window_length",
            "substage",
            "reason_code",
            "metric",
            "observed_value",
            "operator",
            "reference_value",
            "reference_source",
            "detail",
        ]
    ].sort_values(
        [
            "matrix_family",
            "preprocessing",
            "candidate_id",
            "substage",
            "metric",
        ],
        kind="mergesort",
    )


preprocessing_outcomes_df = pca_selection_audit_df.loc[
        pca_selection_audit_df["entity_type"].eq("pca_preprocessing")
        & pca_selection_audit_df["substage"].eq("pareto_selection")
    ].merge(
        pca_preprocessing_summary_df,
        left_on="entity_id",
        right_on="selection_unit_id",
        how="left",
        validate="one_to_one",
    )

dominance_events_df = (
    pca_selection_audit_df.loc[pca_selection_audit_df["substage"].eq("pareto_dominance")]
    .copy()
)

display(pca_selection_stage_summary_df)
display(pca_preprocessing_summary_df)
display(pca_selected_preprocessings_df)
display(audit_funnel_df)
display(candidate_elimination_view.head(expcfg.PCA_MAX_ROWS_TO_DISPLAY))
display(preprocessing_outcomes_df[
        [
            "selection_unit_id",
            "matrix_family",
            "preprocessing",
            "decision",
            "reason_code",
            "selection_status",
            "dominated_by",
        ]
    ])
display(dominance_events_df.head(expcfg.PCA_MAX_ROWS_TO_DISPLAY))
display(pca_selection_audit_checks_df)


,matrix_family,stage,n_entering,n_retained,n_eliminated,retention_rate
0,object_matrix,input,19,19,0,1.000000
1,object_matrix,strict_family_coverage,19,19,0,1.000000
2,object_matrix,complete_pareto_metrics,19,19,0,1.000000
3,object_matrix,pareto_front,19,8,11,0.421053
4,pixel_matrix,input,19,19,0,1.000000
5,pixel_matrix,strict_family_coverage,19,17,2,0.894737
6,pixel_matrix,complete_pareto_metrics,17,17,0,1.000000
7,pixel_matrix,pareto_front,17,6,11,0.352941


,selection_unit_id,matrix_family,preprocessing,preprocessing_steps,sg_window_length,sg_polyorder,wavelength_axis_id,n_candidates,n_expected_variants,n_observed_variants,expected_variants_json,observed_variants_json,missing_variants_json,extra_variants_json,candidate_ids_json,n_accept,n_warning,n_reject,n_blocked_candidates,coverage_complete,all_candidates_admissible,strict_coverage_pass,objective_metrics_complete,preprocessing_eligible,pareto_front,dominated_by,selection_status,class_trace_ratio_min,class_trace_ratio_median,class_trace_ratio_max,class_trace_ratio_iqr,class_trace_ratio_worst,batch_trace_ratio_min,batch_trace_ratio_median,batch_trace_ratio_max,batch_trace_ratio_iqr,batch_trace_ratio_worst,instability_metric_min,instability_metric_median,instability_metric_max,instability_metric_iqr,instability_metric_worst,ncomp_95_min,ncomp_95_median,ncomp_95_max,ncomp_95_iqr,ncomp_95_worst,object_class_trace_ratio_min,object_class_trace_ratio_median,object_class_trace_ratio_max,object_class_trace_ratio_iqr,object_class_trace_ratio_worst,object_batch_trace_ratio_min,object_batch_trace_ratio_median,object_batch_trace_ratio_max,object_batch_trace_ratio_iqr,object_batch_trace_ratio_worst,selection_reason
0,pca_preproc_00077ec72787675de917,object_matrix,snv_sg_smooth,snv+sg_smooth,11.0,2.0,02c5718c409fa484b1b90176a4a1610335c149c0aa6554...,2,2,2,"[""object_mean"", ""object_median""]","[""object_mean"", ""object_median""]",[],[],"[""pca_candidate_75c60cefe42bd0aadc29"", ""pca_ca...",2,0,0,0,True,True,True,True,True,True,,selected,0.321246,0.428176,0.535106,0.106930,0.321246,0.000938,0.001417,0.001896,0.000479,0.001896,0.025675,0.029723,0.033771,0.004048,0.033771,4,5.0,6,1.0,6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,strict_family_coverage;preprocessing_level_par...
1,pca_preproc_13628e4be3540306adda,object_matrix,raw,raw,NaN,NaN,02c5718c409fa484b1b90176a4a1610335c149c0aa6554...,2,2,2,"[""object_mean"", ""object_median""]","[""object_mean"", ""object_median""]",[],[],"[""pca_candidate_4dd7d716213a334a1fa7"", ""pca_ca...",2,0,0,0,True,True,True,True,True,False,pca_preproc_3ff246c42e12fcec72dc;pca_preproc_f...,pareto_dominated,0.096275,0.096447,0.096620,0.000172,0.096275,0.003126,0.004594,0.006062,0.001468,0.006062,0.040446,0.050173,0.059900,0.009727,0.059900,1,1.0,1,0.0,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,dominated_on_preprocessing_pareto_by:pca_prepr...
2,pca_preproc_2faa951879ebe952e1b7,object_matrix,sg_d1,sg_d1,11.0,2.0,02c5718c409fa484b1b90176a4a1610335c149c0aa6554...,2,2,2,"[""object_mean"", ""object_median""]","[""object_mean"", ""object_median""]",[],[],"[""pca_candidate_5d299dc7ab7cec9e3d36"", ""pca_ca...",2,0,0,0,True,True,True,True,True,True,,selected,0.054514,0.057715,0.060916,0.003201,0.054514,0.006212,0.007860,0.009508,0.001648,0.009508,0.009105,0.009895,0.010685,0.000790,0.010685,4,4.5,5,0.5,5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,strict_family_coverage;preprocessing_level_par...
3,pca_preproc_3ff246c42e12fcec72dc,object_matrix,absorbance_sg_smooth,absorbance+sg_smooth,11.0,2.0,02c5718c409fa484b1b90176a4a1610335c149c0aa6554...,2,2,2,"[""object_mean"", ""object_median""]","[""object_mean"", ""object_median""]",[],[],"[""pca_candidate_5c318acb94a8a9018852"", ""pca_ca...",2,0,0,0,True,True,True,True,True,True,,selected,0.118073,0.124489,0.130905,0.006416,0.118073,0.002960,0.004139,0.005318,0.001179,0.005318,0.003657,0.016008,0.028358,0.012350,0.028358,1,1.0,1,0.0,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,strict_family_coverage;preprocessing_level_par...
4,pca_preproc_48eb949137b2d25f73f4,object_matrix,snv,snv,NaN,NaN,02c5718c409fa484b1b90176a4a1610335c149c0aa6554...,2,2,2,"[""object_mean"", ""object_median""]","[""object_mean"", ""object_median""]",[],[],"[""pca_candidate_33d402059c4a36edbd89"", ""pca_ca...",2,0,0,0,True,True,True,True,True,False,pca_preproc_00077ec72787675de917,pareto_dominated,0.320091,0.419863,0.519635,0.099772,0.320091,0.000950,0.001450,0.001949,0.000499,0.001949,0.068938,0.104253,0.139569,0.03531

,selection_unit_id,protocol_hash,input_fingerprint,review_hash,matrix_family,preprocessing,preprocessing_steps,sg_window_length,sg_polyorder,wavelength_axis_id,selection_status,selection_reason
0,pca_preproc_00077ec72787675de917,5d66e659d7da4e69fa647123058bcea08d33fe0155b6c5...,b8f999522dcbf75126a023c6516e6f78a70c99882c20d1...,abff256d5a45b18276840679ebc5497e0b1e0328d1c489...,object_matrix,snv_sg_smooth,snv+sg_smooth,11.0,2.0,02c5718c409fa484b1b90176a4a1610335c149c0aa6554...,selected,strict_family_coverage;preprocessing_level_par...
1,pca_preproc_2faa951879ebe952e1b7,5d66e659d7da4e69fa647123058bcea08d33fe0155b6c5...,b8f999522dcbf75126a023c6516e6f78a70c99882c20d1...,abff256d5a45b18276840679ebc5497e0b1e0328d1c489...,object_matrix,sg_d1,sg_d1,11.0,2.0,02c5718c409fa484b1b90176a4a1610335c149c0aa6554...,selected,strict_family_coverage;preprocessing_level_par...
2,pca_preproc_3ff246c42e12fcec72dc,5d66e659d7da4e69fa647123058bcea08d33fe0155b6c5...,b8f999522dcbf75126a023c6516e6f78a70c99882c20d1...,abff256d5a45b18276840679ebc5497e0b1e0328d1c489...,object_matrix,absorbance_sg_smooth,absorbance+sg_smooth,11.0,2.0,02c5718c409fa484b1b90176a4a1610335c149c0aa6554...,selected,strict_family_coverage;preprocessing_level_par...
3,pca_preproc_4b685ef3705c50f960d3,5d66e659d7da4e69fa647123058bcea08d33fe0155b6c5...,b8f999522dcbf75126a023c6516e6f78a70c99882c20d1...,abff256d5a45b18276840679ebc5497e0b1e0328d1c489...,object_matrix,absorbance,absorbance,NaN,NaN,02c5718c409fa484b1b90176a4a1610335c149c0aa6554...,selected,strict_family_coverage;preprocessing_level_par...
4,pca_preproc_53a12e7fc341de8a6047,5d66e659d7da4e69fa647123058bcea08d33fe0155b6c5...,b8f999522dcbf75126a023c6516e6f78a70c99882c20d1...,abff256d5a45b18276840679ebc5497e0b1e0328d1c489...,object_matrix,absorbance_snv_sg_smooth,absorbance+snv+sg_smooth,11.0,2.0,02c5718c409fa484b1b90176a4a1610335c149c0aa6554...,selected,strict_family_coverage;preprocessing_level_par...
5,pca_preproc_a14496171a4726ed7138,5d66e659d7da4e69fa647123058bcea08d33fe0155b6c5...,b8f999522dcbf75126a023c6516e6f78a70c99882c20d1...,abff256d5a45b18276840679ebc5497e0b1e0328d1c489...,object_matrix,absorbance_snv_sg_d1,absorbance+snv+sg_d1,11.0,2.0,02c5718c409fa484b1b90176a4a1610335c149c0aa6554...,selected,strict_family_coverage;preprocessing_level_par...
6,pca_preproc_e5b6d626093858ae3dde,5d66e659d7da4e69fa647123058bcea08d33fe0155b6c5...,b8f999522dcbf75126a023c6516e6f78a70c99882c20d1...,abff256d5a45b18276840679ebc5497e0b1e0328d1c489...,object_matrix,absorbance_sg_d1,absorbance+sg_d1,11.0,2.0,02c5718c409fa484b1b90176a4a1610335c149c0aa6554...,selected,strict_family_coverage;preprocessing_level_par...
7,pca_preproc_f4b9ecfd6df7f9bb73f0,5d66e659d7da4e69fa647123058bcea08d33fe0155b6c5...,b8f999522dcbf75126a023c6516e6f78a70c99882c20d1...,abff256d5a45b18276840679ebc5497e0b1e0328d1c489...,object_matrix,vector_norm,vector_norm,NaN,NaN,02c5718c409fa484b1b90176a4a1610335c149c0aa6554...,selected,strict_family_coverage;preprocessing_level_par...
8,pca_preproc_67a1ef180e3d2b21de0a,5d66e659d7da4e69fa647123058bcea08d33fe0155b6c5...,b8f999522dcbf75126a023c6516e6f78a70c99882c20d1...,abff256d5a45b18276840679ebc5497e0b1e0328d1c489...,pixel_matrix,sg_smooth,sg_smooth,11.0,2.0,02c5718c409fa484b1b90176a4a1610335c149c0aa6554...,selected,strict_family_coverage;preprocessing_level_par...
9,pca_preproc_a66fb53ff56e1a2cac41,5d66e659d7da4e69fa647123058bcea08d33fe0155b6c5...,b8f999522dcbf75126a023c6516e6f78a70c99882c20d1...,abff256d5a45b18276840679ebc5497e0b1e0328d1c489...,pixel_matrix,absorbance_sg_d2,absorbance+sg_d2,11.0,2.0,02c5718c409fa484b1b90176a4a1610335c149c0aa6554...,selected,strict_family_coverage;preprocessing_level_par...


,entity_type,substage,decision,n_events
0,pca_candidate,artifact_review,eliminated,6
1,pca_candidate,artifact_review,kept,115
2,pca_candidate,artifact_review,warning,12
3,pca_candidate,candidate_admissibility,eliminated,6
4,pca_candidate,candidate_admissibility,kept,127
5,pca_candidate,candidate_generation,entered,133
6,pca_candidate,technical_check,kept,1064
7,pca_candidate,technical_fit_outcome,kept,133
8,pca_preprocessing,pareto_dominance,eliminated,220
9,pca_preprocessing,pareto_metric_completeness,kept,38


,candidate_id,selection_unit_id,matrix_family,matrix_variant,matrix_method,m,balanced_pixel_strategy,preprocessing,sg_window_length,substage,reason_code,metric,observed_value,operator,reference_value,reference_source,detail
4,pca_candidate_535596f1d152aaf4df06,pca_preproc_47057560cb56f3749506,pixel_matrix,balanced_pixels_random_m20,balanced_pixels,20.0,random,absorbance_msc,NaN,artifact_review,critical_artifact,critical_artifact,1.0,==,0.0,PCA human artifact review protocol,review_status=reviewed; review_decision=reject...
10,pca_candidate_535596f1d152aaf4df06,pca_preproc_47057560cb56f3749506,pixel_matrix,balanced_pixels_random_m20,balanced_pixels,20.0,random,absorbance_msc,NaN,candidate_admissibility,candidate_blocked,technical_valid,0.0,==,1.0,PCA candidate admissibility contract,artifact_review_reject;critical_artifact
2,pca_candidate_6806545c4cc2f4c3e112,pca_preproc_47057560cb56f3749506,pixel_matrix,balanced_pixels_random_m10,balanced_pixels,10.0,random,absorbance_msc,NaN,artifact_review,critical_artifact,critical_artifact,1.0,==,0.0,PCA human artifact review protocol,review_status=reviewed; review_decision=reject...
8,pca_candidate_6806545c4cc2f4c3e112,pca_preproc_47057560cb56f3749506,pixel_matrix,balanced_pixels_random_m10,balanced_pixels,10.0,random,absorbance_msc,NaN,candidate_admissibility,candidate_blocked,technical_valid,0.0,==,1.0,PCA candidate admissibility contract,artifact_review_reject;critical_artifact
0,pca_candidate_a06f5bd7028c32d7f767,pca_preproc_47057560cb56f3749506,pixel_matrix,all_pixels,all_pixels,NaN,not_applicable,absorbance_msc,NaN,artifact_review,critical_artifact,critical_artifact,1.0,==,0.0,PCA human artifact review protocol,review_status=reviewed; review_decision=reject...
6,pca_candidate_a06f5bd7028c32d7f767,pca_preproc_47057560cb56f3749506,pixel_matrix,all_pixels,all_pixels,NaN,not_applicable,absorbance_msc,NaN,candidate_admissibility,candidate_blocked,technical_valid,0.0,==,1.0,PCA candidate admissibility contract,artifact_review_reject;critical_artifact
1,pca_candidate_1271e63a08255d8940c3,pca_preproc_870cb19e61da08f82859,pixel_matrix,all_pixels,all_pixels,NaN,not_applicable,msc,NaN,artifact_review,critical_artifact,critical_artifact,1.0,==,0.0,PCA human artifact review protocol,review_status=reviewed; review_decision=reject...
7,pca_candidate_1271e63a08255d8940c3,pca_preproc_870cb19e61da08f82859,pixel_matrix,all_pixels,all_pixels,NaN,not_applicable,msc,NaN,candidate_admissibility,candidate_blocked,technical_valid,0.0,==,1.0,PCA candidate admissibility contract,artifact_review_reject;critical_artifact
3,pca_candidate_1e15b6f9199457673859,pca_preproc_870cb19e61da08f82859,pixel_matrix,balanced_pixels_random_m10,balanced_pixels,10.0,random,msc,NaN,artifact_review,critical_artifact,critical_artifact,1.0,==,0.0,PCA human artifact review protocol,review_status=reviewed; review_decision=reject...
9,pca_candidate_1e15b6f9199457673859,pca_preproc_870cb19e61da08f82859,pixel_matrix,balanced_pixels_random_m10,balanced_pixels,10.0,random,msc,NaN,candidate_admissibility,candidate_blocked,technical_valid,0.0,==,1.0,PCA candidate admissibility contract,artifact_review_reject;critical_artifact


,selection_unit_id,matrix_family,preprocessing,decision,reason_code,selection_status,dominated_by
0,pca_preproc_00077ec72787675de917,object_matrix,snv_sg_smooth,kept,pareto_non_dominated,selected,
1,pca_preproc_13628e4be3540306adda,object_matrix,raw,eliminated,pareto_dominated,pareto_dominated,pca_preproc_3ff246c42e12fcec72dc;pca_preproc_f...
2,pca_preproc_2faa951879ebe952e1b7,object_matrix,sg_d1,kept,pareto_non_dominated,selected,
3,pca_preproc_3ff246c42e12fcec72dc,object_matrix,absorbance_sg_smooth,kept,pareto_non_dominated,selected,
4,pca_preproc_48eb949137b2d25f73f4,object_matrix,snv,eliminated,pareto_dominated,pareto_dominated,pca_preproc_00077ec72787675de917
...,...,...,...,...,...,...,...
33,pca_preproc_c842bc180f715772d124,pixel_matrix,snv,eliminated,pareto_dominated,pareto_dominated,pca_preproc_a66fb53ff56e1a2cac41;pca_preproc_a...
34,pca_preproc_c9595becbcd00e0c33d5,pixel_matrix,vector_norm,kept,pareto_non_dominated,selected,
35,pca_preproc_e9fa191d8552156b6ad0,pixel_matrix,absorbance_sg_smooth,kept,pareto_non_dominated,selected,
36,pca_preproc_ecfbcfb732618d61e72e,pixel_matrix,sg_d1,eliminated,pareto_dominated,pareto_dominated,pca_preproc_67a1ef180e3d2b21de0a


,stage,substage,entity_type,entity_id,related_entity_id,track_id,decision,reason_code,metric,observed_value,operator,reference_value,reference_source,mechanism,detail
1824,03,pareto_dominance,pca_preprocessing,pca_preproc_13628e4be3540306adda,pca_preproc_3ff246c42e12fcec72dc,,eliminated,pareto_dominated_by,class_trace_ratio,0.096275,<=,0.118073,pca_preproc_3ff246c42e12fcec72dc,pareto,goal=maximize; strict_improvement=True
1825,03,pareto_dominance,pca_preprocessing,pca_preproc_13628e4be3540306adda,pca_preproc_f7e37b0208bb3542553e,,eliminated,pareto_dominated_by,class_trace_ratio,0.096275,<=,0.096326,pca_preproc_f7e37b0208bb3542553e,pareto,goal=maximize; strict_improvement=True
1826,03,pareto_dominance,pca_preprocessing,pca_preproc_48eb949137b2d25f73f4,pca_preproc_00077ec72787675de917,,eliminated,pareto_dominated_by,class_trace_ratio,0.320091,<=,0.321246,pca_preproc_00077ec72787675de917,pareto,goal=maximize; strict_improvement=True
1827,03,pareto_dominance,pca_preprocessing,pca_preproc_640069b0b3ffb4548bee,pca_preproc_00077ec72787675de917,,eliminated,pareto_dominated_by,class_trace_ratio,0.127118,<=,0.321246,pca_preproc_00077ec72787675de917,pareto,goal=maximize; strict_improvement=True
1828,03,pareto_dominance,pca_preprocessing,pca_preproc_640069b0b3ffb4548bee,pca_preproc_857a431511524dd63f9d,,eliminated,pareto_dominated_by,class_trace_ratio,0.127118,<=,0.159038,pca_preproc_857a431511524dd63f9d,pareto,goal=maximize; strict_improvement=True
1829,03,pareto_dominance,pca_preprocessing,pca_preproc_640069b0b3ffb4548bee,pca_preproc_a14496171a4726ed7138,,eliminated,pareto_dominated_by,class_trace_ratio,0.127118,<=,0.220404,pca_preproc_a14496171a4726ed7138,pareto,goal=maximize; strict_improvement=True
1830,03,pareto_dominance,pca_preprocessing,pca_preproc_640069b0b3ffb4548bee,pca_preproc_e5b6d626093858ae3dde,,eliminated,pareto_dominated_by,class_trace_ratio,0.127118,<=,0.273084,pca_preproc_e5b6d626093858ae3dde,pareto,goal=maximize; strict_improvement=True
1831,03,pareto_dominance,pca_preprocessing,pca_preproc_640069b0b3ffb4548bee,pca_preproc_f474f06165792caceb9f,,eliminated,pareto_dominated_by,class_trace_ratio,0.127118,<=,0.282694,pca_preproc_f474f06165792caceb9f,pareto,goal=maximize; strict_improvement=True
1832,03,pareto_dominance,pca_preprocessing,pca_preproc_7b7c6743d820a7f1a8e5,pca_preproc_00077ec72787675de917,,eliminated,pareto_dominated_by,class_trace_ratio,0.197838,<=,0.321246,pca_preproc_00077ec72787675de917,pareto,goal=maximize; strict_improvement=True
1833,03,pareto_dominance,pca_preprocessing,pca_preproc_7b7c6743d820a7f1a8e5,pca_preproc_53a12e7fc341de8a6047,,eliminated,pareto_dominated_by,class_trace_ratio,0.197838,<=,0.465585,pca_preproc_53a12e7fc341de8a6047,pareto,goal=maximize; strict_improvement=True


,check,passed,detail
0,audit_schema_exact,True,"missing=[], extra=[]"
1,audit_column_order_matches_contract,True,"observed=['stage', 'substage', 'entity_type', ..."
2,audit_natural_keys_unique,True,duplicates=0
3,audit_stage_is_notebook03,True,observed=['03']
4,pca_audit_is_track_independent,True,nonempty_track_rows=0
...,...,...,...
44,pareto_dominance_stays_within_matrix_family,True,invalid_rows=[]
45,pareto_dominance_numeric_relations_hold,True,failures=[]
46,pareto_dominance_has_at_least_one_strict_objec...,True,non_strict_pairs=[]
47,pareto_dominance_event_contract,True,invalid_rows=[]


## 8. Gel des artefacts et revalidation après lecture disque

In [8]:
score_column_audit_df = assert_no_forbidden_score_columns(
    {
        "pca_candidate_registry": pca_candidate_registry_df,
        "pca_summary": pca_summary_df,
        "pca_scoring_diagnostics": pca_scoring_diagnostics_df,
        "pca_preprocessing_summary": pca_preprocessing_summary_df,
        "pca_selected_preprocessings": pca_selected_preprocessings_df,
        "pca_artifact_review": pca_artifact_review_df,
        "pca_selection_audit": pca_selection_audit_df,
    }
)
if any("confirmation" in column.lower() for column in pca_scoring_diagnostics_df.columns):
    raise RuntimeError("External-batch diagnostics are forbidden in notebook 03.")

save_parquet(pca_candidate_registry_df, OUTPUT_PATHS["candidate_registry"])
save_parquet(pca_summary_df, OUTPUT_PATHS["summary"])
save_parquet(pca_scoring_diagnostics_df, OUTPUT_PATHS["diagnostics"])
save_parquet(pca_preprocessing_summary_df, OUTPUT_PATHS["preprocessing_summary"])
save_parquet(pca_selected_preprocessings_df, OUTPUT_PATHS["selected"])
save_parquet(pca_artifact_review_df, OUTPUT_PATHS["artifact_review"])
save_parquet(pca_selection_audit_df, OUTPUT_PATHS["selection_audit"])

saved_candidate_registry_df = load_parquet(OUTPUT_PATHS["candidate_registry"])
saved_preprocessing_summary_df = load_parquet(OUTPUT_PATHS["preprocessing_summary"])
saved_shortlist_df = load_parquet(OUTPUT_PATHS["selected"])
saved_selection_audit_df = load_parquet(OUTPUT_PATHS["selection_audit"])

validate_pca_preprocessing_shortlist(
    saved_shortlist_df,
    max_per_family=expcfg.MAX_PCA_PREPROCESSINGS_PER_FAMILY,
    expected_families=expcfg.PCA_SELECTION_EXPECTED_FAMILIES,
    expected_protocol_hash=PROTOCOL_HASH,
    expected_input_fingerprint=INPUT_FINGERPRINT,
    expected_review_hash=review_hash
)

saved_audit_checks_df = assert_pca_selection_audit_consistency(
    saved_selection_audit_df,
    candidate_registry=saved_candidate_registry_df,
    preprocessing_summary=saved_preprocessing_summary_df,
    selected_preprocessings=saved_shortlist_df,
    strict=True,
)

missing_outputs = [str(path) for path in OUTPUT_PATHS.values() if not path.exists()]
if missing_outputs:
    raise RuntimeError(
        "Some PCA output artifacts are missing after saving: "
        + ", ".join(missing_outputs)
    )

output_hashes_df = pd.DataFrame(
    [
        {"artifact": name, "path": str(path), "sha256": sha256_file(path)}
        for name, path in OUTPUT_PATHS.items()
    ]
)

display(score_column_audit_df)
display(saved_audit_checks_df)
display(output_hashes_df)

print(
    "PCA selection units:",
    saved_shortlist_df["selection_unit_id"].nunique(),
)
print("protocol_hash:", PROTOCOL_HASH)
print("run_fingerprint:", RUN_FINGERPRINT)
print("results directory:", RESULTS_DIR.resolve())


,table,n_columns,forbidden_score_columns,score_free
0,pca_candidate_registry,14,,True
1,pca_summary,12,,True
2,pca_scoring_diagnostics,20,,True
3,pca_preprocessing_summary,58,,True
4,pca_selected_preprocessings,12,,True
5,pca_artifact_review,16,,True
6,pca_selection_audit,15,,True


,check,passed,detail
0,audit_schema_exact,True,"missing=[], extra=[]"
1,audit_column_order_matches_contract,True,"observed=['stage', 'substage', 'entity_type', ..."
2,audit_natural_keys_unique,True,duplicates=0
3,audit_stage_is_notebook03,True,observed=['03']
4,pca_audit_is_track_independent,True,nonempty_track_rows=0
...,...,...,...
44,pareto_dominance_stays_within_matrix_family,True,invalid_rows=[]
45,pareto_dominance_numeric_relations_hold,True,failures=[]
46,pareto_dominance_has_at_least_one_strict_objec...,True,non_strict_pairs=[]
47,pareto_dominance_event_contract,True,invalid_rows=[]


,artifact,path,sha256
0,candidate_registry,C:\Users\alixg\OneDrive - Université Paris-Dau...,a07382603c958b868a2b8068c130fe46ecdc15eacdf225...
1,summary,C:\Users\alixg\OneDrive - Université Paris-Dau...,6dadf481b7903bf36b44c383960f4c2ef59673420dbf45...
2,diagnostics,C:\Users\alixg\OneDrive - Université Paris-Dau...,9d286b77ae444b88dff7be488c82919c6f1b372a046dd4...
3,preprocessing_summary,C:\Users\alixg\OneDrive - Université Paris-Dau...,7f76e5d5770d93a7e81b00cea5ccd2c96bfb7bf8a811dc...
4,selected,C:\Users\alixg\OneDrive - Université Paris-Dau...,2718313510b86bc097db068a094801d2d89783217b0c52...
5,artifact_review,C:\Users\alixg\OneDrive - Université Paris-Dau...,7570f2963dfc27c809ff08bff4b9d0d36c577c495aaac7...
6,visual_review,C:\Users\alixg\OneDrive - Université Paris-Dau...,8d504d1ffeb3fa3b18fec1e960174803dbc6b1d7e3caf5...
7,selection_audit,C:\Users\alixg\OneDrive - Université Paris-Dau...,f5b57249ed3b92d12499ce78576637cbd53f5d0a41796b...


PCA selection units: 14
protocol_hash: 5d66e659d7da4e69fa647123058bcea08d33fe0155b6c59d71001820dbc78f9e
run_fingerprint: ef11b61ee355079c1fd13e01f780f601e48d56eb9be1e1a3c1f1e334ce5674b6
results directory: C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\03_pca_8tracks_v5_px_qc_v1
